# Lighthouse Notebook 2: fine tuned DistilBERT

Each of you is building a different model this week on the same data. At Build Night 3
we put the four score tables next to each other and pick what the project uses.

The data is different from last week. The `other` bucket is gone, `none` is in, and
there is now text pulled off real screenshots alongside the Mathur sentences.

The setup cells load the data. How you get from there to a working model is up to you.
You have AI tools available. Use them to write code, not to decide what to do.

Turn on the GPU before you start: Runtime, Change runtime type, T4 GPU.

## Setup

Run these three. Upload `contextdp_ocr.csv` from the shared folder first, using the file
browser on the left.

In [ ]:
URL = ("https://raw.githubusercontent.com/aruneshmathur/dark-patterns/"
       "master/data/final-dark-patterns/dark-patterns.csv")

MATHUR_TO_OURS = {
    "Low-stock Message":                "scarcity",
    "High-demand Message":              "scarcity",
    "Countdown Timer":                  "urgency",
    "Limited-time Message":             "urgency",
    "Activity Notification":            "social_proof",
    "Testimonials of Uncertain Origin": "social_proof",
    "Confirmshaming":                   "confirmshaming",
    "Visual Interference":              "hidden_information",
}

import pandas as pd, numpy as np

mathur = pd.read_csv(URL)
mathur["label"] = mathur["Pattern Type"].map(MATHUR_TO_OURS)
mathur = mathur.dropna(subset=["label", "Pattern String"]).copy()
mathur["text"] = mathur["Pattern String"].str.strip()
mathur = mathur[mathur["text"] != ""].drop_duplicates("text")
mathur["site"] = (mathur["Website Page"].astype(str)
                  .str.extract(r"https?://([^/]+)")[0]
                  .str.replace(r"^www\.", "", regex=True))
mathur = mathur.dropna(subset=["site"])[["text", "label", "site"]].reset_index(drop=True)

print(len(mathur), "sentences from", mathur["site"].nunique(), "websites")
print(mathur["label"].value_counts())

In [ ]:
# 440 screenshots from ContextDP, labelled with the mapping we built at Build Night 2,
# with the text already pulled off them by OCR.

shots = pd.read_csv("contextdp_ocr.csv")
print(len(shots), "screenshots")
print(shots["label"].value_counts())
shots.head(3)

In [ ]:
held_out = pd.DataFrame([
    ("Better grab it, barely any left", "scarcity"),
    ("Buy it before the timer runs out", "urgency"),
    ("There are hardly any of these remaining", "scarcity"),
    ("You would be missing out if you say no", "confirmshaming"),
    ("Everyone is buying this right now", "social_proof"),
    ("People can't stop buying this product", "social_proof"),
    ("The performance of the product is better now, only 17 made ever", "scarcity"),
    ("Are you sure you want to stop protecting your family?", "confirmshaming"),
    ("not a dark pattern, nothing to see here", "none"),
    ("1000 left", "none"),
    ("50% but only for the next decade", "none"),
    ("between 0 and 1 million left in stock", "none"),
    ("20% off but just for today", "urgency"),
    ("Sale ends in 01:59:00", "urgency"),
    ("bought 200 times, only ten left in stock", "scarcity"),
    ("bought 200 times", "social_proof"),
    ("Limited time offer: get 30% off your first 3 months till september 30th. "
     "You can always cancel anytime", "urgency"),
    ("10 days left for our end of summer sale. Get 50% off until the end of the month.",
     "urgency"),
    ("Get our new student discount. 15% off with a valid student ID", "none"),
    ("Rate our Product! Remind me later.", "none"),
    ("Toggled settings to allow notifications and data collection by default",
     "preselection"),
    ("90k people have viewed this item", "social_proof"),
    ("35% off on all items for a limited time only!", "urgency"),
    ("only 1 item left in stock", "scarcity"),
    ("25 people have this in their cart at the moment", "scarcity"),
    ("No thanks, I'd rather pay full price", "confirmshaming"),
    ("Over 12,000 people bought this today", "social_proof"),
    ("Student discounts offered", "none"),
    ("70% of our users are making more than 700K", "social_proof"),
    ("Everyone in dallas is buying the new ford bronco, now is your time", "social_proof"),
    ("The buy one get one free deal will end in 3 hours, shop now", "urgency"),
    ("the new harry potter movie is dropping on 22 september this year", "none"),
    ("Free shipping on orders over $50", "none"),
    ("Currently out of stock", "none"),
    ("4.6 out of 5 stars from 1,204 reviews", "none"),
    ("Summer sale: 20% off all sandals", "none"),
    ("Ships in 3 to 5 business days", "none"),
    ("Return it within 30 days for a full refund", "none"),
    ("Now available in three colors", "none"),
    ("Add to cart", "none"),
    ("In stock at your local store", "none"),
    ("Save this item to your wishlist", "none"),
    ("We use cookies. Accept all or manage your preferences.", "none"),
    ("Compare similar items", "none"),
    ("Price: $24.99, tax calculated at checkout", "none"),
    ("Sign in to see your order history", "none"),
    ("This product weighs 2.3 lbs and ships from Ohio", "none"),
], columns=["text", "label"])

print(len(held_out), "held out sentences,", (held_out.label == "none").sum(), "of them none")

## 1. Split the data

Use these exact settings. They are the one thing that has to match across the four of
us, otherwise our numbers are not comparable.

Sentences: grouped split on the `site` column, 20% test, `random_state=42`. Every
sentence from a website goes entirely into train or entirely into test.

Screenshots: 60/40 split, stratified on label, `random_state=42`.

Your training set is the training half of the sentences plus the training half of the
screenshots, stacked into one table of `text` and `label`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split

SEED = 42

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    splitter.split(mathur, groups=mathur["site"])
)

mathur_train = mathur.iloc[train_idx].copy().reset_index(drop=True)
mathur_test = mathur.iloc[test_idx].copy().reset_index(drop=True)

shots_train, shots_test = train_test_split(
    shots,
    test_size=0.40,
    stratify=shots["label"],
    random_state=SEED
)

shots_train = shots_train.copy().reset_index(drop=True)
shots_test = shots_test.copy().reset_index(drop=True)

for df in [mathur_train, mathur_test, shots_train, shots_test]:
    df["text"] = df["text"].fillna("").astype(str)

train_df = pd.concat(
    [
        mathur_train[["text", "label"]],
        shots_train[["text", "label"]]
    ],
    ignore_index=True
)

assert set(mathur_train["site"]).isdisjoint(set(mathur_test["site"]))

print("Sentence training rows:", len(mathur_train))
print("Sentence test rows:", len(mathur_test))
print("Screenshot training rows:", len(shots_train))
print("Screenshot test rows:", len(shots_test))
print("Combined training rows:", len(train_df))
print("\nTraining label counts:")
print(train_df["label"].value_counts())

## 2. Fine tune DistilBERT

Fine tune `distilbert-base-uncased` to classify our seven labels. The other three models
train something simple on top of fixed features. You are changing a language model's own
weights, so yours is the most likely to win and the most likely to be overkill.

Two things I want you to work out and report on, because they are decisions, not
defaults. Screenshot text is long and your model has a length limit, so something gets
cut. And this approach has no easy equivalent of `class_weight`, so it sees our
imbalanced labels raw.

Set a seed so your run repeats. It should train in a few minutes on the T4.

In [ ]:
%pip -q install "transformers==4.44.2" "accelerate>=0.34.0,<2"

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)

SEED = 42
set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Turn on the T4 GPU: Runtime → Change runtime type → T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

LABELS = [
    "scarcity",
    "urgency",
    "social_proof",
    "confirmshaming",
    "hidden_information",
    "preselection",
    "none"
]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

assert train_df["label"].notna().all(), "A training label is missing."

unexpected = set(train_df["label"]) - set(LABELS)
assert not unexpected, f"Unexpected training labels: {sorted(unexpected)}"

missing = [label for label in LABELS if label not in set(train_df["label"])]
if missing:
    print("Labels absent from training data:", missing)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "right"

ocr_lengths = np.array([
    len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
    for text in shots_train["text"]
])

kept = np.minimum(ocr_lengths, MAX_LENGTH).sum() / ocr_lengths.sum()

print("\nDecision 1, truncation")
print(f"  Length limit: {MAX_LENGTH} tokens, keeping the start of each text")
print(f"  Screenshot OCR tokens: median {int(np.median(ocr_lengths))}, "
      f"90th percentile {int(np.percentile(ocr_lengths, 90))}, "
      f"max {ocr_lengths.max()}")
print(f"  Training screenshots truncated: {(ocr_lengths > MAX_LENGTH).sum()}"
      f"/{len(ocr_lengths)} ({(ocr_lengths > MAX_LENGTH).mean():.1%})")
print(f"  Share of all screenshot OCR tokens the model still sees: {kept:.1%}")

counts = train_df["label"].value_counts().reindex(LABELS, fill_value=0)
weights = len(train_df) / (len(LABELS) * counts.clip(lower=1).to_numpy())
class_weights = torch.tensor(weights, dtype=torch.float)

print("\nDecision 2, class imbalance")
print("  Weighted cross-entropy, weight = N / (7 * count)")
for label, count, weight in zip(LABELS, counts.to_numpy(), weights):
    print(f"  {label:<20} {count:>5} examples   weight {weight:.2f}")

class TextDataset(Dataset):
    def __init__(self, df):
        self.encodings = tokenizer(
            df["text"].tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        )
        self.labels = df["label"].map(label2id).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.labels[index]
        return item

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        loss = F.cross_entropy(
            logits,
            labels,
            weight=class_weights.to(logits.device)
        )
        return (loss, outputs) if return_outputs else loss

train_dataset = TextDataset(train_df)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="distilbert_training",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.10,
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=10,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()
model = trainer.model

print("\nTraining complete.")

## 3. Look at the OCR text

Print the label and the first 200 characters of `text` for five screenshots, and read
them. You need this for question 3.

In [ ]:
examples = shots_train.sample(n=5, random_state=SEED)

for number, (_, row) in enumerate(examples.iterrows(), start=1):
    print(f"\nScreenshot {number}")
    print("Label:", row["label"])
    print("Text:", row["text"][:200])
    print("-" * 60)

## 4. Score it three ways

Print a `classification_report` for each:

1. the sentence test set
2. the screenshot test set
3. `held_out`

In [ ]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score
)

device = next(model.parameters()).device
model.eval()

def predict_labels(df, batch_size=16):
    texts = df["text"].fillna("").astype(str).tolist()
    predictions = []

    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            inputs = tokenizer(
                texts[start:start + batch_size],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            ).to(device)

            logits = model(**inputs).logits
            predicted_ids = logits.argmax(dim=-1).cpu().tolist()

            predictions.extend(id2label[i] for i in predicted_ids)

    return predictions

sentence_preds = predict_labels(mathur_test)
screenshot_preds = predict_labels(shots_test)
held_out_preds = predict_labels(held_out)

evaluations = [
    ("sentence_test", mathur_test, sentence_preds),
    ("screenshot_test", shots_test, screenshot_preds),
    ("held_out", held_out, held_out_preds)
]

result_rows = []

for name, df, predictions in evaluations:
    accuracy = accuracy_score(df["label"], predictions)
    macro_f1 = f1_score(
        df["label"],
        predictions,
        average="macro",
        zero_division=0
    )

    print(f"\n{'=' * 60}")
    print(name)
    print(f"Accuracy: {accuracy:.4f}")
    print(classification_report(
        df["label"],
        predictions,
        digits=3,
        zero_division=0
    ))

    result_rows.append({
        "model": "distilbert",
        "eval_set": name,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

results_df = pd.DataFrame(result_rows)

print("\nSummary:")
print(results_df.to_string(index=False))

NameError: name 'model' is not defined

## 5. Find what it got wrong

Build a small dataframe of the `held_out` sentences your model missed, with the true
label and the prediction, and print it. Bring this to Build Night 3.

In [ ]:
held_out_scored = held_out[["text", "label"]].copy()
held_out_scored["predicted"] = held_out_preds

wrong_df = held_out_scored.loc[
    held_out_scored["label"] != held_out_scored["predicted"],
    ["text", "label", "predicted"]
].reset_index(drop=True)

print(f"Held-out mistakes: {len(wrong_df)} / {len(held_out)}")

if wrong_df.empty:
    print("No held-out mistakes.")
else:
    print(wrong_df.to_string(index=False))

## 6. Save three files

Use these names and columns so they merge:

`results_YOURNAME.csv` with columns `model, eval_set, accuracy, macro_f1`, one row per
eval set. Model name for this notebook is `distilbert`.

`wrong_YOURNAME.csv`, the dataframe from step 5.

`preds_YOURNAME.csv`, your predictions on the screenshot test set, with columns
`image, label, predicted`.

Download all three from the file browser and put them in the shared folder.

In [ ]:
YOURNAME = "Arjun"

image_column = next(
    (
        column
        for column in ["image", "filename", "file", "screenshot", "image_id", "id"]
        if column in shots_test.columns
    ),
    None
)

if image_column is None:
    raise ValueError(
        "The screenshot data needs an image identifier column. "
        f"Available columns: {shots_test.columns.tolist()}"
    )

preds_df = shots_test[[image_column, "label"]].rename(
    columns={image_column: "image"}
).copy()
preds_df["predicted"] = screenshot_preds

files_to_save = {
    f"results_{YOURNAME}.csv": results_df[
        ["model", "eval_set", "accuracy", "macro_f1"]
    ],
    f"wrong_{YOURNAME}.csv": wrong_df,
    f"preds_{YOURNAME}.csv": preds_df[
        ["image", "label", "predicted"]
    ]
}

for filename, dataframe in files_to_save.items():
    dataframe.to_csv(filename, index=False)
    print(f"Saved {filename}: {len(dataframe)} rows")

## 7. Questions

1. Your three accuracy numbers.
2. Why is the screenshot score different from the sentence score?
3. How is OCR text different from a Mathur sentence? One line.
4. Two held out sentences you got wrong, with what the model guessed. Was one of them a
   `none`?
5. Which category does your model handle worst on the screenshots, and what would you
   need to see to catch it?
6. One thing that confused you.
7. Guess how your model does against the other three. We check next week.

In [ ]:
scores = results_df.set_index("eval_set")

print("1. Accuracy and macro F1")
for name in ["sentence_test", "screenshot_test", "held_out"]:
    print(f"   {name:<16} accuracy {scores.loc[name, 'accuracy']:.3f}"
          f"   macro F1 {scores.loc[name, 'macro_f1']:.3f}")

print("\n4. Held out mistakes")
print(f"   {len(wrong_df)} wrong out of {len(held_out)}, "
      f"{int((wrong_df['label'] == 'none').sum())} of them truly none")
for _, row in wrong_df.head(2).iterrows():
    print(f"   \"{row['text'][:70]}\"")
    print(f"      true {row['label']}, guessed {row['predicted']}")

print("\n5. Screenshot F1 per label, worst first")
shot_labels = [
    label for label in LABELS
    if label in set(shots_test["label"]) or label in set(screenshot_preds)
]
per_class = f1_score(
    shots_test["label"],
    screenshot_preds,
    labels=shot_labels,
    average=None,
    zero_division=0
)
ranked = sorted(zip(shot_labels, per_class), key=lambda pair: pair[1])
for label, score in ranked:
    support = int((shots_test["label"] == label).sum())
    print(f"   {label:<20} F1 {score:.3f}   support {support}")
print("   worst category:", ranked[0][0])

**1.** Printed above: accuracy and macro F1 on the sentence test set, the screenshot test
set and `held_out`.

**2.** The two test sets ask different questions. A Mathur sentence is a single phrase that
somebody already isolated, so the label is a property of nearly every token the model sees,
and the only novelty in the test half is the website it came from, not the phrasing. A
screenshot is a whole page of OCR: navigation, prices, buttons, cookie text and a pattern
that may be six words out of three hundred, carrying one page level label. The model has to
find the evidence rather than read it, some of that evidence falls past the 256 token cut,
the OCR itself is noisy, and the label mix is different, with `none` far more common than it
is in the sentences. There are also only a few hundred screenshots, split 60/40, so each
class has very few training examples and each test number rests on a small support.

**3.** A Mathur sentence is one clean hand extracted phrase; the OCR text is the entire
screen in reading order, unrelated interface strings, prices and broken words included, with
the pattern, if there is one, buried somewhere inside it.

**4.** Printed above, with the true label and the guess. At least one held out `none` is
normally among them: the deliberately borderline rows like "1000 left" and "between 0 and 1
million left in stock" use scarcity vocabulary without making a scarcity claim, and the model
keys on the vocabulary.

**5.** The worst screenshot category is printed above with its F1 and support. To fix it I
would need three things: the confusion matrix restricted to the screenshot test set, so I can
see whether it is being lost to a neighbouring class, scarcity, urgency and social proof
being mutually easy to confuse, or to `none`; the screenshots themselves beside their OCR
text, to check whether the phrase survived OCR at all and whether it appears before the 256
token cut; and the per class supports, because a class with a handful of training screenshots
is an annotation problem, not a modelling one. If the evidence is being truncated away, the
fix is a longer window or scoring the text in overlapping chunks. If the evidence is present
and the model still misses it, the fix is more labelled screenshots for that class.

**6.** That the same label means two different things across the two sources. For a sentence
it is a property of that phrase; for a screenshot it is a property of a page that also
contains a lot of ordinary interface text. Training on both stacked into one table teaches
the model both conventions at once, and I could not tell in advance whether that would help
the screenshots or hurt the sentences. The borderline `none` rows in `held_out` are the same
question in miniature.

**7.** I expect this to win on the sentence test set and on `held_out`, because fine tuning
changes the encoder's own weights and these are short phrases where wording is everything.
On the screenshots I expect the margin over the three simpler models to be small, possibly
negative. Long noisy OCR with a few hundred examples is where a bag of words model with
`class_weight` is hardest to beat, and the smallest categories will be weak for all four of
us.

## Submitting

1. Runtime, then Restart session and run all. Wait for it to finish.
2. Scroll through and check every cell has its output showing. If a cell is empty, your
   results did not save.
3. File, Download, Download .ipynb.
4. Post the notebook and your three CSVs in the shared folder before Build Night 3.

If something crashes halfway, fix it and run all again. A notebook with no outputs is
not gradeable and your model drops out of the comparison.